# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya
This notebook provides a template for loading and exploring the FAIR² Croissant dataset using the `mlcroissant` library.

### Dataset Source
This dataset is defined via a Croissant schema accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

The dataset includes ordered logistic regression output tables and survey data on rangeland management knowledge adoption in Northern Kenya.

In [ ]:
# Ensure the required mlcroissant package is installed!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load metadata and dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and the fields within each record set.

All record sets, fields, and columns are referenced by their `@id` as per the Croissant schema.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.metadata.record_set
recordset_ids = []
if record_sets is not None:
    for rs in record_sets:
        print(f"Record Set: {rs['@id']} (name: {rs.get('name', '')})")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields/@id:")
        for field in fields:
            print(f"    - {field['@id']}")
        recordset_ids.append(rs['@id'])
else:
    print("No record sets found in this dataset. Exploring records programmatically...")
    # As a fallback, use dataset.record_set property (if exists)
    # Some datasets don't expose record_set as top-level; we'll attempt to infer.
    # List available record sets using the .record_sets() function
    recordset_ids = dataset.record_sets()
    if not recordset_ids:
        from pprint import pprint
        print("Available record sets via dataset.record_sets():")
        pprint(recordset_ids)
    else:
        for rs_id in recordset_ids:
            print(f"Record Set: {rs_id}")

## 3. Data Extraction
Load data from a record set by its `@id` and inspect the fields. We'll load each record set that is available in the dataset. All references use `@id`.

In [ ]:
# Collect available record set @idsrecord_sets_to_load = []
if len(recordset_ids) > 0:
    record_sets_to_load = recordset_ids
else:
    # As a fallback, try common record set ids (domain knowledge or documentation required)
    print("No explicit record set ids discovered; please refer to documentation for available record sets.")

dataframes = {}
for record_set_id in record_sets_to_load:
    print(f"Loading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Fields in {record_set_id}:")
    print(f"    {list(df.columns)}")
    print(df.head(2))

# Select one record set for EDA (use the first available, if present)if record_sets_to_load:
    main_record_set_id = record_sets_to_load[0]
    print(f"\nMain record set for EDA: {main_record_set_id}")
    print(f"Fields: {list(dataframes[main_record_set_id].columns)}")
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter records, normalize numeric fields, and group data. All columns and fields are referenced using their `@id`.

In [ ]:
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Pick a numeric field (use domain knowledge or by checking dtypes)
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Attempt to find a numeric field (float or int) by dtype
numeric_field_id = None
for col in df.columns:
    if np.issubdtype(df[col].dropna().dtype, np.number):
        numeric_field_id = col
        break
if numeric_field_id is None:
    print('No numeric field found. Please update the field selection.')
else:
    print(f'Numeric field selected: {numeric_field_id}')

threshold = df[numeric_field_id].mean() if numeric_field_id else None
if numeric_field_id:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt grouping by a likely categorical field
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id:
            if df[col].dtype == 'object' and df[col].nunique() > 1 and df[col].nunique() < (len(df)//2):
                group_field_id = col
                break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped filtered data by '{group_field_id}' and mean {numeric_field_id}:")
        print(grouped_df.head())
    else:
        print('No suitable categorical field found for grouping.')

## 5. Visualization
Visualize the distribution of the selected numeric field and its grouping, referencing fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_field_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30, color='b')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook has demonstrated:
- How to load a FAIR² dataset using its Croissant schema and the `mlcroissant` library.
- How to inspect record sets and fields using `@id` references, ensuring clear entity referencing per the schema.
- How to extract, filter, normalize, and visualize tabular data aligned to the dataset's semantics.
- Next steps could include domain-specific analyses such as statistical testing on adoption predictors or geospatial mapping using provided regional attributes.